### Load and Process JSON Documents

In [1]:
from langchain.document_loaders import JSONLoader

loader = JSONLoader(file_path='./rag_docs/wikidata_rag_demo.jsonl',
                    jq_schema='.',
                    text_content=False,
                    json_lines=True)
wiki_docs = loader.load()

In [2]:
len(wiki_docs)

1801

In [3]:
wiki_docs[3]

Document(metadata={'source': 'C:\\Users\\91900\\Desktop\\ML-projects\\Contextual-RAG-System-with-Hybrid-Search-and-Reranking\\rag_docs\\wikidata_rag_demo.jsonl', 'seq_num': 4}, page_content='{"id": "71548", "title": "Chi-square distribution", "paragraphs": ["In probability theory and statistics, the chi-square distribution (also chi-squared or formula_1\\u00a0 distribution) is one of the most widely used theoretical probability distributions. Chi-square distribution with formula_2 degrees of freedom is written as formula_3. It is a special case of gamma distribution.", "Chi-square distribution is primarily used in statistical significance tests and confidence intervals. It is useful, because it is relatively easy to show that certain probability distributions come close to it, under certain conditions. One of these conditions is that the null hypothesis must be true. Another one is that the different random variables (or observations) must be independent of each other."]}')

In [4]:
import json
from langchain.docstore.document import Document    # Class for storing a piece of text and associated Metadata

wiki_docs_processed = []

for doc in wiki_docs:
    doc = json.loads(doc.page_content)
    metadata = {
        "title": doc['title'],
        "id": doc['id'],
        "source": "Wikipedia",
        "page": 1
    }
    data = ' '.join(doc['paragraphs'])
    wiki_docs_processed.append(Document(page_content=data, metadata=metadata))

In [5]:
wiki_docs_processed[3]

Document(metadata={'title': 'Chi-square distribution', 'id': '71548', 'source': 'Wikipedia', 'page': 1}, page_content='In probability theory and statistics, the chi-square distribution (also chi-squared or formula_1\xa0 distribution) is one of the most widely used theoretical probability distributions. Chi-square distribution with formula_2 degrees of freedom is written as formula_3. It is a special case of gamma distribution. Chi-square distribution is primarily used in statistical significance tests and confidence intervals. It is useful, because it is relatively easy to show that certain probability distributions come close to it, under certain conditions. One of these conditions is that the null hypothesis must be true. Another one is that the different random variables (or observations) must be independent of each other.')

### Load and Process PDF documents

In [6]:

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import uuid

def create_contextual_chunks(file_path, chunk_size=3500, chunk_overlap=0):
    print("Loading pages:", file_path)

    loader = PyMuPDFLoader(file_path)
    doc_pages = loader.load()

    print("Chunking pages:", file_path)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    doc_chunks = splitter.split_documents(doc_pages)

    contextual_chunks = []

    for chunk in doc_chunks:
        metadata = {
            "id": str(uuid.uuid4()),
            "page": chunk.metadata.get("page", 0),
            "source": chunk.metadata.get("source", file_path),
            "title": file_path.replace("\\", "/").split("/")[-1]
        }

        contextual_chunks.append(
            Document(
                page_content=chunk.page_content,
                metadata=metadata
            )
        )

    print(f"Created {len(contextual_chunks)} chunks from: {file_path}")
    return contextual_chunks


In [7]:
from glob import glob

pdf_files = glob('./rag_docs/*.pdf')
pdf_files

['./rag_docs\\attention_paper.pdf',
 './rag_docs\\cnn_paper.pdf',
 './rag_docs\\resnet_paper.pdf',
 './rag_docs\\vision_transformer.pdf']

In [8]:
paper_docs = []

for fp in pdf_files:
    paper_docs.extend(create_contextual_chunks(file_path=fp, chunk_size=3500))

Loading pages: ./rag_docs\attention_paper.pdf
Chunking pages: ./rag_docs\attention_paper.pdf
Created 16 chunks from: ./rag_docs\attention_paper.pdf
Loading pages: ./rag_docs\cnn_paper.pdf
Chunking pages: ./rag_docs\cnn_paper.pdf
Created 11 chunks from: ./rag_docs\cnn_paper.pdf
Loading pages: ./rag_docs\resnet_paper.pdf
Chunking pages: ./rag_docs\resnet_paper.pdf
Created 24 chunks from: ./rag_docs\resnet_paper.pdf
Loading pages: ./rag_docs\vision_transformer.pdf
Chunking pages: ./rag_docs\vision_transformer.pdf
Created 28 chunks from: ./rag_docs\vision_transformer.pdf


In [9]:
len(paper_docs)

79

In [10]:
paper_docs[0]

Document(metadata={'id': '7f244249-d242-4fab-b8eb-dbe5a5380c15', 'page': 0, 'source': './rag_docs\\attention_paper.pdf', 'title': 'attention_paper.pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and de

### Combine all document chunks in one list

In [11]:
len(wiki_docs_processed)

1801

In [12]:
total_docs = wiki_docs_processed + paper_docs
len(total_docs)

1880

### Vector DB Indexing for Semantic Search

In [13]:

from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer

class LocalEmbeddings(Embeddings):
    def __init__(self):
        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

embeddings = LocalEmbeddings()
print("Local embedding model loaded successfully!")


c:\Users\91900\Desktop\ML-projects\Contextual-RAG-System-with-Hybrid-Search-and-Reranking\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Local embedding model loaded successfully!


In [14]:
from langchain_chroma import Chroma

# create vector DB of docs and embeddings - takes < 30s on Colab
chroma_db = Chroma.from_documents(documents=total_docs,
                                  collection_name='my_context_db',
                                  embedding=embeddings,
                                  # need to set the Distance function to Cosine else it uses Euclidean by default
                                  # check https://docs.trychroma.com/guides#changing-the-distance-function
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory="./my_context_db")              # set custom disk location

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Batches: 100%|██████████| 59/59 [01:42<00:00,  1.73s/it]


In [15]:

results = chroma_db.similarity_search(
    "What is the attention mechanism in transformers?",
    k=3
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:500])


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



--- Result 1 ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-dista

--- Result 2 ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the att

### Load Vector DB from disk

This is just to show once you have a vector database on disk you can just load and create a connection to it anytime

In [16]:
# load from disk
chroma_db = Chroma(persist_directory="./my_context_db",
                   collection_name='my_context_db',
                   embedding_function=embeddings)

print("local vectorDB loaded successfully!")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


local vectorDB loaded successfully!


In [17]:
chroma_db

### Semantic Similarity based Retrieval

We use simple cosine similarity here and retrieve the top 5 similar documents based on the user input query

In [18]:
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 5})

print("retriever created successfully!")

retriever created successfully!


### BM25 Indexing for Keyword based Retrieval

In [19]:
from langchain.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents=total_docs,
                                              k=5)
bm25_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001A1A61E8A10>, k=5)

## Build Retrieval Strategy

### Build Base Ensemble Retriever

In [20]:
from langchain.retrievers import EnsembleRetriever

# Reciprocal Rank Fusion:

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, similarity_retriever],      # Vector Similarity Search + BM25 Search
    weights=[0.5, 0.5]
)
ensemble_retriever

EnsembleRetriever(retrievers=[BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001A1A61E8A10>, k=5), VectorStoreRetriever(tags=['Chroma', 'LocalEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001A1992582C0>, search_kwargs={'k': 5})], weights=[0.5, 0.5])

### Chained Retrieval with Reranker

In [21]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

# download an open-source Re-Ranker Model - BAAI/bge-reranker-v2-m3
reranker = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker_compressor = CrossEncoderReranker(model=reranker, top_n=5)

# Retriever 2 - Uses a Reranker model to rerank retrieval results from the previous retriever
final_retriever = ContextualCompressionRetriever(
    base_retriever=ensemble_retriever,              # Stage 1: Base Retriever: (Vector Similarity Search + BM25 Search) : Reciprocal Rank Fusion
    base_compressor=reranker_compressor             # Stage 2: Re-ranker
)
final_retriever

ContextualCompressionRetriever(base_compressor=CrossEncoderReranker(model=HuggingFaceCrossEncoder(client=<sentence_transformers.cross_encoder.CrossEncoder.CrossEncoder object at 0x000001A19E3EF980>, model_name='BAAI/bge-reranker-v2-m3', model_kwargs={}), top_n=5), base_retriever=EnsembleRetriever(retrievers=[BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001A1A61E8A10>, k=5), VectorStoreRetriever(tags=['Chroma', 'LocalEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001A1992582C0>, search_kwargs={'k': 5})], weights=[0.5, 0.5]))

In [22]:
from IPython.display import display, Markdown

def display_docs(docs):
    for doc in docs:
        print('Metadata:', doc.metadata)
        print('Content Brief:')
        display(Markdown(doc.page_content[:1000]))
        print()

In [23]:
query = "what is machine learning?"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

Metadata: {'id': '564928', 'page': 1, 'source': 'Wikipedia', 'title': 'Machine learning'}
Content Brief:


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


Metadata: {'id': '651629', 'page': 1, 'source': 'Wikipedia', 'title': 'Arthur Samuel'}
Content Brief:


Arthur Lee Samuel (December 5, 1901 – July 29, 1990) was an American pioneer in the field of computer gaming and artificial intelligence. He coined the term "machine learning" in 1959. The Samuel Checkers-playing Program was among the world's first successful self-learning programs, and as such a very early demonstration of the fundamental concept of artificial intelligence (AI). He was also a senior member in the TeX community who devoted much time giving personal attention to the needs of users and wrote an early TeX manual in 1983.


Metadata: {'id': '359370', 'page': 1, 'source': 'Wikipedia', 'title': 'Supervised learning'}
Content Brief:


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data.


Metadata: {'title': 'Artificial intelligence', 'id': '6360', 'source': 'Wikipedia', 'page': 1}
Content Brief:


Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is also a field of study which tries to make computers "smart". They work on their own without being encoded with commands. John McCarthy came up with the name "Artificial Intelligence" in 1955. In general use, the term "artificial intelligence" means a programme which mimics human cognition. At least some of the things we associate with other minds, such as learning and problem solving can be done by computers, though not in the same way as we do. Andreas Kaplan and Michael Haenlein define AI as a system’s ability to correctly interpret external data, to learn from such data, and to use those learnings to achieve specific goals and tasks through flexible adaptation. An ideal (perfect) intelligent machine is a flexible agent which perceives its environment and takes actions to maximize its chance of success at some goal or objective. As machines become increasingly capable, mental facu


Metadata: {'title': 'Reinforcement learning', 'id': '610032', 'source': 'Wikipedia', 'page': 1}
Content Brief:


Reinforcement learning (RL) is teaching a "software agent" how to behave in an environment by telling it how good it's doing. It is an area of machine learning inspired by behaviorist psychology. Reinforcement learning is different from supervised learning because the correct inputs and outputs are never shown. Also, reinforcement learning usually learns as it goes (online learning) unlike supervised learning. This means an agent has to choose between exploring and sticking with what it knows best. A reinforcement learning system is made of a "policy" (formula_1), a "reward function" (formula_2), a "value function" (formula_3), and an optional "model" of the environment.

In [24]:
query = "what is the difference between transformers and vision transformers?"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

Metadata: {'id': '864980ac-60de-4b7b-b028-3bc0fd1f72a4', 'page': 0, 'source': './rag_docs\\vision_transformer.pdf', 'title': 'vision_transformer.pdf'}
Content Brief:


Published as a conference paper at ICLR 2021
AN IMAGE IS WORTH 16X16 WORDS:
TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗,
Xiaohua Zhai∗, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer,
Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby∗,†
∗equal technical contribution, †equal advising
Google Research, Brain Team
{adosovitskiy, neilhoulsby}@google.com
ABSTRACT
While the Transformer architecture has become the de-facto standard for natural
language processing tasks, its applications to computer vision remain limited. In
vision, attention is either applied in conjunction with convolutional networks, or
used to replace certain components of convolutional networks while keeping their
overall structure in place. We show that this reliance on CNNs is not necessary
and a pure transformer applied directly to sequences of image patches can perform
very well on image classiﬁcation tasks. When pre-traine


Metadata: {'id': 'c2aa04de-df31-4155-a9aa-f56a35350dc3', 'page': 2, 'source': './rag_docs\\vision_transformer.pdf', 'title': 'vision_transformer.pdf'}
Content Brief:


Published as a conference paper at ICLR 2021
Transformer Encoder
MLP 
Head
Vision Transformer (ViT)
*
Linear Projection of Flattened Patches
* Extra learnable
     [ cl ass]  embedding
1
2
3
4
5
6
7
8
9
0
Patch + Position 
Embedding
Class
Bird
Ball
Car
...
Embedded 
Patches
Multi-Head 
Attention
Norm
MLP
Norm
+
L x
+
Transformer Encoder
Figure 1: Model overview. We split an image into ﬁxed-size patches, linearly embed each of them,
add position embeddings, and feed the resulting sequence of vectors to a standard Transformer
encoder. In order to perform classiﬁcation, we use the standard approach of adding an extra learnable
“classiﬁcation token” to the sequence. The illustration of the Transformer encoder was inspired by
Vaswani et al. (2017).
3
METHOD
In model design we follow the original Transformer (Vaswani et al., 2017) as closely as possible.
An advantage of this intentionally simple setup is that scalable NLP Transformer architectures – and
their efﬁcient implementations – can b


Metadata: {'id': '487af159-8b94-4ebf-b70e-ba4c6c3b6662', 'page': 4, 'source': './rag_docs\\vision_transformer.pdf', 'title': 'vision_transformer.pdf'}
Content Brief:


Published as a conference paper at ICLR 2021
Model
Layers
Hidden size D
MLP size
Heads
Params
ViT-Base
12
768
3072
12
86M
ViT-Large
24
1024
4096
16
307M
ViT-Huge
32
1280
5120
16
632M
Table 1: Details of Vision Transformer model variants.
We also evaluate on the 19-task VTAB classiﬁcation suite (Zhai et al., 2019b). VTAB evaluates
low-data transfer to diverse tasks, using 1 000 training examples per task. The tasks are divided into
three groups: Natural – tasks like the above, Pets, CIFAR, etc. Specialized – medical and satellite
imagery, and Structured – tasks that require geometric understanding like localization.
Model Variants. We base ViT conﬁgurations on those used for BERT (Devlin et al., 2019), as
summarized in Table 1. The “Base” and “Large” models are directly adopted from BERT and we
add the larger “Huge” model. In what follows we use brief notation to indicate the model size and
the input patch size: for instance, ViT-L/16 means the “Large” variant with 16×16 input patch siz


Metadata: {'title': 'Errors and residuals in statistics', 'id': '188232', 'source': 'Wikipedia', 'page': 1}
Content Brief:


Statistical errors and residuals occur because measurement is never exact. It is not possible to do an exact measurement, but it is possible to say how accurate a measurement is. One can measure the same thing again and again, and collect all the data together. This allows us to do statistics on the data. What is meant by errors and residuals is the difference between the observed or measured value and the real value, which is unknown. If there is only one random variable, the difference between statistical errors and residuals is the difference between the mean of the population against the mean of the (observed) sample. In that case the residual is the difference between what the probability distribution says, and what was actually measured.


Metadata: {'title': 'Bose-Einstein statistics', 'id': '114994', 'source': 'Wikipedia', 'page': 1}
Content Brief:


In statistical mechanics, Bose-Einstein statistics means the statistics of a system where you can not tell the difference between any of the particles, and the particles are bosons. Bosons are fundamental particles like the photon. The Bose-Einstein distribution tells you how many particles have a certain energy. The formula is with formula_2 and where:

## Build the RAG Pipeline

In [25]:
from langchain_core.prompts import ChatPromptTemplate


rag_prompt = """Answer the question in 2-3 sentences using only the context.
Start with a direct answer. Do not summarize the paper.
If the context does not contain the answer, say you do not know.

Context:
{context}

Question:
{question}

Answer:"""


rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

In [26]:
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:3b", temperature=0)

def format_docs(docs):    # formats the document content into a structured string
    return "\n\n".join(doc.page_content for doc in docs)

qa_rag_chain = (
    {
        "context": (final_retriever           # Revokes the Retriever using the user Query
                      |
                    format_docs),             # formats the document content into a structured string
        "question": RunnablePassthrough()     # to directly pass the user Query without any modifications
    }
      |
    rag_prompt_template
      |
     llm
)

In [27]:
from IPython.display import display, Markdown

query = "What is machine learning?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Machine learning is the study and construction of algorithms that can learn and make predictions on data.

In [28]:
print(rag_prompt_template.messages[0].prompt.template)

Answer the question in 2-3 sentences using only the context.
Start with a direct answer. Do not summarize the paper.
If the context does not contain the answer, say you do not know.

Context:
{context}

Question:
{question}

Answer:


In [29]:
query = "What is a Vision Transformer?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

A Vision Transformer is a self-attention-based architecture that applies a standard Transformer directly to images, with the fewest possible modifications, by splitting an image into patches, linearly embedding each patch, adding position embeddings, and feeding the resulting sequence of vectors to a standard Transformer encoder.

In [30]:
test_cases = [
    {
        "question": "What is machine learning?",
        "expected": "Computers learn from data without being explicitly programmed."
    },
    {
        "question": "What is a Vision Transformer?",
        "expected": "It splits an image into patches, embeds them, and processes them with a Transformer encoder."
    },
    {
        "question": "How does the Transformer decoder prevent attending to future positions?",
        "expected": "It uses a causal mask to prevent each position from attending to future positions."
    },
    {
        "question": "What is the role of self-attention in the Transformer encoder?",
        "expected": "Each position can attend to other positions in the input representation."
    }
]

for i, case in enumerate(test_cases, 1):
    print(f"\n{'=' * 60}")
    print(f"Test {i}: {case['question']}")
    print(f"Expected key idea: {case['expected']}")

    result = qa_rag_chain.invoke(case["question"])
    print(f"Generated answer: {result.content}")


Test 1: What is machine learning?
Expected key idea: Computers learn from data without being explicitly programmed.
Generated answer: Machine learning is the study and construction of algorithms that can learn and make predictions on data.

Test 2: What is a Vision Transformer?
Expected key idea: It splits an image into patches, embeds them, and processes them with a Transformer encoder.
Generated answer: A Vision Transformer is a self-attention-based architecture that applies a standard Transformer directly to images, with the fewest possible modifications, by splitting an image into patches, linearly embedding each patch, adding position embeddings, and feeding the resulting sequence of vectors to a standard Transformer encoder.

Test 3: How does the Transformer decoder prevent attending to future positions?
Expected key idea: It uses a causal mask to prevent each position from attending to future positions.
Generated answer: The Transformer decoder prevents attending to future posi

In [31]:
import json

evaluation_results = [
    {
        "question": "What is machine learning?",
        "expected_key_idea": "Computers learn from data without being explicitly programmed.",
        "answer": "Machine learning is a subfield of computer science that gives computers the ability to learn without being explicitly programmed."
    },
    {
        "question": "What is a Vision Transformer?",
        "expected_key_idea": "It splits an image into patches, embeds them, and processes them with a Transformer encoder.",
        "answer": "A Vision Transformer (ViT) is a type of neural network architecture that uses self-attention mechanisms to process visual data, such as images. It is similar to the Transformer architecture used in natural language processing, but is designed to handle 2D images instead of 1D sequences. The ViT architecture consists of a sequence of patches extracted from the image, which are then fed into a Transformer encoder."
    },
    {
        "question": "How does the Transformer decoder prevent attending to future positions?",
        "expected_key_idea": "It uses a causal mask to prevent each position from attending to future positions.",
        "answer": "The Transformer decoder prevents attending to future positions by masking out (setting to −∞) all values in the input of the softmax function that correspond to illegal connections."
    },
    {
        "question": "What is the role of self-attention in the Transformer encoder?",
        "expected_key_idea": "Each position can attend to other positions in the input representation.",
        "answer": "Self-attention is used in the Transformer encoder to allow each position in the encoder to attend to all positions in the previous layer of the encoder."
    }
]

with open("rag_evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=2, ensure_ascii=False)

print("Saved 4 evaluation examples to rag_evaluation_results.json")

Saved 4 evaluation examples to rag_evaluation_results.json


In [32]:
retrieval_test_questions = [
    "What is machine learning?",
    "What is a Vision Transformer?",
    "How does the Transformer decoder prevent attending to future positions?",
    "What is the role of self-attention in the Transformer encoder?"
]

for i, question in enumerate(retrieval_test_questions, 1):
    docs = final_retriever.invoke(question)

    print(f"\n{'=' * 60}")
    print(f"Question {i}: {question}")

    for rank, doc in enumerate(docs, 1):
        print(
            f"{rank}. {doc.metadata.get('title', 'Unknown')} "
            f"| Page: {doc.metadata.get('page', 'Unknown')}"
        )
        print(doc.page_content[:250].replace("\n", " "))


Question 1: What is machine learning?
1. Machine learning | Page: 1
Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and constru
2. Arthur Samuel | Page: 1
Arthur Lee Samuel (December 5, 1901 – July 29, 1990) was an American pioneer in the field of computer gaming and artificial intelligence. He coined the term "machine learning" in 1959. The Samuel Checkers-playing Program was among the world's first s
3. Supervised learning | Page: 1
In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work wit
4. cnn_paper.pdf | Page: 0
An Introduction to Convolutional Neural Networks Keiron O’Shea1 and Ryan Nash2 1 Department

In [33]:
question = "How does the Transformer decoder prevent attending to future positions?"

docs = chroma_db.similarity_search(question, k=5)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Vector Search Result {i} ---")
    print("Page:", doc.metadata.get("page", "Unknown"))
    print(doc.page_content[:700].replace("\n", " "))


--- Vector Search Result 1 ---
Page: 2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position- wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer

--- Vector Search Result 2 ---
Page: 2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left a

In [34]:
for doc in total_docs:
    text = doc.page_content.lower()

    if (
        "subsequent positions" in text
        or "masking mechanism" in text
        or "illegal connections" in text
    ):
        print("\n--- Matching Chunk ---")
        print("Source:", doc.metadata.get("title", "Unknown"))
        print("Page:", doc.metadata.get("page", "Unknown"))
        print(doc.page_content[:1200])


--- Matching Chunk ---
Source: attention_paper.pdf
Page: 2
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1
Encoder and Decoder Stacks
Encoder:
The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder:
The dec

In [35]:
for doc in total_docs:
    text = doc.page_content.lower()

    if "illegal connections" in text:
        print("Source:", doc.metadata.get("title", "Unknown"))
        print("Page:", doc.metadata.get("page", "Unknown"))
        print("\nFull chunk:\n")
        print(doc.page_content)
        break

Source: attention_paper.pdf
Page: 4

Full chunk:

output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O
where headi = Attention(QW Q
i , KW K
i , V W V
i )
Where the projections are parameter matrices W Q
i
∈Rdmodel×dk, W K
i
∈Rdmodel×dk, W V
i
∈Rdmodel×dv
and W O ∈Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3
Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the

In [36]:
question = "How does the Transformer decoder prevent attending to future positions?"

docs = chroma_db.similarity_search(question, k=10)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc.metadata.get("title", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))

    if "illegal connections" in doc.page_content.lower():
        print(">>> EXACT MASKING EXPLANATION FOUND <<<")

    print(doc.page_content[:300].replace("\n", " "))


--- Result 1 ---
Source: attention_paper.pdf
Page: 2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder

--- Result 2 ---
Source: attention_paper.pdf
Page: 2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder

--- Result 3 ---
Source: attention_paper.pdf
Page: 1
entirely on self-attention to compute representations of its input and output without using sequence- aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate self-attention and discuss its adva

In [37]:
question = "How does the Transformer decoder prevent attending to future positions?"

docs = bm25_retriever.invoke(question)

for i, doc in enumerate(docs, 1):
    print(f"\n--- BM25 Result {i} ---")
    print("Source:", doc.metadata.get("title", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))

    if "illegal connections" in doc.page_content.lower():
        print(">>> EXACT MASKING EXPLANATION FOUND <<<")

    print(doc.page_content[:400].replace("\n", " "))


--- BM25 Result 1 ---
Source: attention_paper.pdf
Page: 2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The f

--- BM25 Result 2 ---
Source: attention_paper.pdf
Page: 4
>>> EXACT MASKING EXPLANATION FOUND <<<
output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O where headi = Attention(QW Q i , KW K i

--- BM25 Result 3 ---
Source: Logistic 

In [38]:
# Retrieve more candidates from both search methods
similarity_retriever.search_kwargs = {"k": 10}
bm25_retriever.k = 10

# Rebuild hybrid retrieval
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, similarity_retriever],
    weights=[0.5, 0.5]
)

# Rebuild reranking pipeline
final_retriever = ContextualCompressionRetriever(
    base_retriever=ensemble_retriever,
    base_compressor=reranker_compressor
)

print("Updated hybrid retrieval: 10 candidates per retriever")

Updated hybrid retrieval: 10 candidates per retriever


In [39]:
question = "How does the Transformer decoder prevent attending to future positions?"

docs = final_retriever.invoke(question)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Reranked Result {i} ---")
    print("Source:", doc.metadata.get("title", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))
    print(doc.page_content[:400].replace("\n", " "))


--- Reranked Result 1 ---
Source: attention_paper.pdf
Page: 2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The f

--- Reranked Result 2 ---
Source: attention_paper.pdf
Page: 4
output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O where headi = Attention(QW Q i , KW K i

--- Reranked Result 3 ---
Source: vision_transformer.pdf
Page: 7
Publis

In [40]:
question = "How does the Transformer decoder prevent attending to future positions?"

docs = final_retriever.invoke(question)

for i, doc in enumerate(docs, 1):
    text = doc.page_content.lower()

    print(f"\n--- Result {i} ---")
    print("Source:", doc.metadata.get("title", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))

    if "illegal connections" in text:
        print("EXACT MASKING EXPLANATION FOUND")
        start = text.find("illegal connections")
        print(doc.page_content[max(0, start - 500):start + 700])
    else:
        print(doc.page_content[:200].replace("\n", " "))


--- Result 1 ---
Source: attention_paper.pdf
Page: 2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, 

--- Result 2 ---
Source: attention_paper.pdf
Page: 4
EXACT MASKING EXPLANATION FOUND
osition in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward
information flow in the decoder to preserve the auto-regressive property. We implement this
inside of scaled dot-product attention by masking out (setting to −∞) all values in the input
of the softmax which correspond to illegal connections. See Figure 2.
3.3
Position-wise Feed-Forward Networks
In addition to attention sub-layers, each of the layers in our encoder and decoder c

In [41]:
question = "How does the Transformer decoder prevent attending to future positions?"

response = qa_rag_chain.invoke(question)
print(response.content)

The Transformer decoder prevents attending to future positions by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.


In [42]:
question = "Who won the FIFA World Cup in 1998?"

response = qa_rag_chain.invoke(question)
print(response.content)

I do not know.


In [43]:
evaluation_questions = [
    {
        "question": "What is machine learning?",
        "expected_key_ideas": ["learn without being explicitly programmed"]
    },
    {
        "question": "What is a Vision Transformer?",
        "expected_key_ideas": [
            "image patches",
            "transformer encoder",
            "self-attention"
        ]
    },
    {
        "question": "How does the Transformer decoder prevent attending to future positions?",
        "expected_key_ideas": [
            "masking",
            "illegal connections",
            "negative infinity",
            "softmax"
        ]
    },
    {
        "question": "How is self-attention used in the Transformer encoder?",
        "expected_key_ideas": [
            "each position",
            "attend to all positions"
        ]
    },
    {
        "question": "Who won the FIFA World Cup in 1998?",
        "expected_key_ideas": ["I do not know"]
    }
]

print(f"Evaluation questions: {len(evaluation_questions)}")

Evaluation questions: 5


In [44]:
import json

evaluation_results = []

for item in evaluation_questions:
    question = item["question"]

    response = qa_rag_chain.invoke(question)
    answer = response.content

    result = {
        "question": question,
        "expected_key_ideas": item["expected_key_ideas"],
        "generated_answer": answer
    }

    evaluation_results.append(result)

    print("\n" + "=" * 80)
    print("Question:", question)
    print("Expected key ideas:", item["expected_key_ideas"])
    print("Generated answer:", answer)

with open("rag_evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=2, ensure_ascii=False)

print("\nSaved results to rag_evaluation_results.json")


Question: What is machine learning?
Expected key ideas: ['learn without being explicitly programmed']
Generated answer: Machine learning is a subfield of computer science that gives computers the ability to learn without being explicitly programmed.

Question: What is a Vision Transformer?
Expected key ideas: ['image patches', 'transformer encoder', 'self-attention']
Generated answer: A Vision Transformer (ViT) is a type of neural network architecture that uses self-attention mechanisms to process visual data, such as images. It is similar to the Transformer architecture used in natural language processing, but is designed to handle 2D images by dividing them into fixed-size patches, linearly embedding each patch, and feeding the resulting sequence of vectors to a standard Transformer encoder.

Question: How does the Transformer decoder prevent attending to future positions?
Expected key ideas: ['masking', 'illegal connections', 'negative infinity', 'softmax']
Generated answer: The Tr

In [45]:
import json
from pathlib import Path

results_path = Path("rag_evaluation_results.json")

with open(results_path, "r", encoding="utf-8") as f:
    saved_results = json.load(f)

print("File exists:", results_path.exists())
print("Number of saved results:", len(saved_results))
print("First question:", saved_results[0]["question"])

File exists: True
Number of saved results: 5
First question: What is machine learning?


In [46]:
retrieval_methods = {
    "Vector Search": similarity_retriever,
    "BM25": bm25_retriever,
    "Hybrid Search": ensemble_retriever,
    "Hybrid + Reranking": final_retriever
}

retrieval_comparison = []

for item in evaluation_questions:
    question = item["question"]

    print("\n" + "=" * 80)
    print("QUESTION:", question)

    for method_name, retriever in retrieval_methods.items():
        docs = retriever.invoke(question)

        result = {
            "question": question,
            "method": method_name,
            "retrieved_count": len(docs),
            "sources": [
                f"{doc.metadata.get('title', 'Unknown')} | "
                f"page {doc.metadata.get('page', 'Unknown')}"
                for doc in docs
            ]
        }

        retrieval_comparison.append(result)

        print(f"\n{method_name} — {len(docs)} documents")
        for rank, doc in enumerate(docs, 1):
            print(
                f"  {rank}. {doc.metadata.get('title', 'Unknown')} "
                f"| page {doc.metadata.get('page', 'Unknown')}"
            )

import json

with open("rag_retrieval_comparison.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_comparison, f, indent=2, ensure_ascii=False)

print("\nSaved to rag_retrieval_comparison.json")


QUESTION: What is machine learning?

Vector Search — 10 documents
  1. Machine learning | page 1
  2. Machine learning | page 1
  3. Supervised learning | page 1
  4. Supervised learning | page 1
  5. Arthur Samuel | page 1
  6. Arthur Samuel | page 1
  7. Deep learning | page 1
  8. Deep learning | page 1
  9. Artificial intelligence | page 1
  10. Artificial intelligence | page 1

BM25 — 10 documents
  1. Errors and residuals in statistics | page 1
  2. Causality | page 1
  3. Artificial intelligence | page 1
  4. Sui generis | page 1
  5. cnn_paper.pdf | page 0
  6. Reinforcement learning | page 1
  7. Syllogism | page 1
  8. Fourier transform | page 1
  9. Bayesian network | page 1
  10. Billboard Hot 100 | page 1

Hybrid Search — 14 documents
  1. Artificial intelligence | page 1
  2. Machine learning | page 1
  3. Supervised learning | page 1
  4. Arthur Samuel | page 1
  5. Deep learning | page 1
  6. Errors and residuals in statistics | page 1
  7. Causality | page 1
  8. Sui 

In [47]:
question = "What is the main idea behind the Transformer attention mechanism?"
retrieved_docs = final_retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, 1):
    print(
        i,
        "Source:", doc.metadata.get("title", "Unknown"),
        "| Page:", doc.metadata.get("page", "Unknown")
    )

1 Source: attention_paper.pdf | Page: 1
2 Source: attention_paper.pdf | Page: 2
3 Source: attention_paper.pdf | Page: 4
4 Source: attention_paper.pdf | Page: 8
5 Source: vision_transformer.pdf | Page: 0


In [48]:
question = "What is machine learning?"

for method, retriever in {
    "Vector Search": similarity_retriever,
    "BM25": bm25_retriever,
    "Hybrid Search": ensemble_retriever,
    "Hybrid + Reranking": final_retriever
}.items():
    docs = retriever.invoke(question)

    print(f"\n{'=' * 50}\n{method}")
    for rank, doc in enumerate(docs[:5], 1):
        print(
            f"\n{rank}. {doc.metadata.get('title', 'Unknown')}"
            f" | Page {doc.metadata.get('page', 'Unknown')}"
        )
        print(doc.page_content[:200].replace("\n", " "))


Vector Search

1. Machine learning | Page 1
Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligenc

2. Machine learning | Page 1
Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligenc

3. Supervised learning | Page 1
In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these

4. Supervised learning | Page 1
In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these

5. Arthur Samuel | Page 1
Arthur Lee Samuel (December 

In [49]:

question = "What is machine learning?"

retrievers_to_check = {
    "Vector Search": similarity_retriever,
    "BM25": bm25_retriever,
    "Hybrid Search": ensemble_retriever,
    "Hybrid + Reranking": final_retriever
}

for method, retriever in retrievers_to_check.items():
    docs = retriever.invoke(question)

    print(f"\n{'=' * 60}")
    print(method)

    for rank, doc in enumerate(docs[:5], 1):
        print(f"\nRank {rank}: {doc.metadata.get('title', 'Unknown')}")
        print(doc.page_content[:250].replace("\n", " "))



Vector Search

Rank 1: Machine learning
Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and constru

Rank 2: Machine learning
Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and constru

Rank 3: Supervised learning
In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work wit

Rank 4: Supervised learning
In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of t

In [50]:
relevance_judgments = {
    "What is machine learning?": {
        "Machine learning",
        "Arthur Samuel",
        "Deep learning",
        "Supervised learning",
        "Artificial neural network"
    },
    "What is a Vision Transformer?": {
        "vision_transformer.pdf"
    },
    "How does the Transformer decoder prevent attending to future positions?": {
        "attention_paper.pdf"
    },
    "How is self-attention used in the Transformer encoder?": {
        "attention_paper.pdf"
    },
    "Who won the FIFA World Cup in 1998?": set()
}

print("Relevance judgments defined for", len(relevance_judgments), "questions.")

Relevance judgments defined for 5 questions.


In [51]:

import json

with open("rag_retrieval_comparison.json", "r", encoding="utf-8") as f:
    retrieval_results = json.load(f)

def is_relevant(question, source):
    relevant_titles = relevance_judgments[question]
    return source in relevant_titles

print(f"{'Question':55} {'Method':25} {'Relevant':>8} {'P@5':>6}")
print("-" * 100)

for result in retrieval_results:
    question = result["question"]
    method = result["method"]
    top5 = result["sources"][:5]

    relevant_count = 0

    for source in top5:
        title = source.split(" | page ")[0]

        if is_relevant(question, title):
            relevant_count += 1

    precision = relevant_count / 5

    print(
        f"{question[:53]:55} "
        f"{method:25} "
        f"{relevant_count:>8} "
        f"{precision:>6.2f}"
    )


Question                                                Method                    Relevant    P@5
----------------------------------------------------------------------------------------------------
What is machine learning?                               Vector Search                    5   1.00
What is machine learning?                               BM25                             0   0.00
What is machine learning?                               Hybrid Search                    4   0.80
What is machine learning?                               Hybrid + Reranking               4   0.80
What is a Vision Transformer?                           Vector Search                    5   1.00
What is a Vision Transformer?                           BM25                             1   0.20
What is a Vision Transformer?                           Hybrid Search                    4   0.80
What is a Vision Transformer?                           Hybrid + Reranking               5   1.00
How does the Tran

In [52]:

question = "How does the Transformer decoder prevent attending to future positions?"

for method, retriever in {
    "Vector Search": similarity_retriever,
    "BM25": bm25_retriever,
    "Hybrid Search": ensemble_retriever,
    "Hybrid + Reranking": final_retriever
}.items():
    docs = retriever.invoke(question)

    print(f"\n{'=' * 65}")
    print(method)

    for rank, doc in enumerate(docs[:5], 1):
        print(f"\nRank {rank}: {doc.metadata.get('title', 'Unknown')}")
        print(f"Page: {doc.metadata.get('page', 'Unknown')}")
        print(doc.page_content[:700].replace("\n", " "))



Vector Search

Rank 1: attention_paper.pdf
Page: 2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position- wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer

Rank 2: attention_paper.pdf
Page: 2
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in t

In [53]:
for i, doc in enumerate(retrieved_docs[:2], 1):
    print(f"\n--- Top Result {i} ---")
    print(doc.page_content[:1500])


--- Top Result 1 ---
1
Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational eff

In [54]:
test_questions = [
    "What is the main idea behind the Transformer attention mechanism?",
    "What is the role of self-attention in the Transformer encoder?",
    "How does the Transformer decoder prevent attending to future positions?",
    "What is a Vision Transformer?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 70}")
    print(f"Question {i}: {question}")

    result = qa_rag_chain.invoke(question)
    print("\nAnswer:")
    print(result.content)


Question 1: What is the main idea behind the Transformer attention mechanism?

Answer:
The main idea behind the Transformer attention mechanism is to jointly attend to information from different representation subspaces at different positions, allowing the model to jointly attend to information from different parts of the input sequence simultaneously.

Question 2: What is the role of self-attention in the Transformer encoder?

Answer:
Self-attention is a mechanism used in the Transformer encoder to allow each position in the input sequence to attend to all positions in the same sequence, including those that are far away from it. This allows the model to jointly attend to information from different representation subspaces at different positions, which is particularly useful for modeling long-range dependencies in the input sequence.

Question 3: How does the Transformer decoder prevent attending to future positions?

Answer:
The Transformer decoder prevents attending to future posit

In [55]:
query = "What is NLP and its relation to linguistics?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

NLP is a field in Artificial Intelligence that aims to program computers to automatically understand and generate human languages. It is closely related to linguistics, as it seeks to understand the structure and usage of human languages, and to apply this understanding to create more effective language processing systems.

In [56]:
query = "What is the difference between AI, ML and DL?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

I do not know.

In [57]:
query = "What is the difference between transformers and vision transformers?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Transformers and Vision Transformers (ViT) are both self-attention-based architectures, but they differ in their application and design. 

Transformers were originally designed for natural language processing tasks, where they have become the de-facto standard. They are typically pre-trained on large text corpora and then fine-tuned on specific tasks.

Vision Transformers (ViT), on the other hand, were designed specifically for computer vision tasks, such as image classification. They apply the Transformer architecture directly to sequences of image patches, without the need for convolutional networks. This allows ViT to perform well on image recognition tasks, even with smaller computational resources.

In [58]:
query = "How is self-attention important in transformers?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Self-attention is important in transformers because it allows the model to attend to all positions in the input sequence simultaneously, rather than relying on a sequential or convolutional approach. This enables the model to capture long-range dependencies and relationships in the input data, which is particularly useful for tasks such as image classification and natural language processing.

In [59]:
query = "How does a resnet work?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

A ResNet works by using residual connections to ease the optimization difficulty of deep networks, allowing them to overcome the vanishing gradient problem and achieve higher accuracy when the depth increases.